# Hybrid choice model

Jointly estimate a binary choice, a structural latent-attitude equation, and a continuous measurement equation.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import pandas as pd
import torch

from IPython.display import HTML, display

from torchdcm import (
    Beta,
    ChoiceDataset,
    ChoiceLatentEffect,
    ContinuousIndicator,
    HybridChoiceModel,
    LatentVariable,
    UtilitySpec,
)

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
rng = np.random.default_rng(22)
n_obs = 160
z = rng.normal(size=n_obs)
x_b = rng.normal(size=n_obs)
attitude = 0.60 * z + rng.normal(scale=0.70, size=n_obs)
utility_b = 0.20 + 0.70 * x_b + 0.60 * attitude
p_b = 1.0 / (1.0 + np.exp(-utility_b))
choice = np.where(rng.uniform(size=n_obs) < p_b, "B", "A")

frame = pd.DataFrame(
    {
        "choice": choice,
        "x_a": np.zeros(n_obs),
        "x_b": x_b,
        "z": z,
        "attitude": attitude,
    }
)
data = ChoiceDataset.from_wide(
    frame,
    alternatives=["A", "B"],
    choice="choice",
    variables={"x": {"A": "x_a", "B": "x_b"}},
    obs_variables={"z": "z", "attitude": "attitude"},
)


In [3]:
spec = UtilitySpec()
spec.utility("A", Beta("ASC_A", init=0.0, fixed=True))
spec.utility(
    "B",
    Beta("ASC_B", init=0.0)
    + Beta("B_X", init=0.50) * "x",
)

latent_variables = [
    LatentVariable(
        "ATTITUDE",
        intercept=0.0,
        coefficients={"z": Beta("GAMMA_Z", init=0.30)},
        sigma_init=1.0,
        sigma_fixed=True,
    )
]
choice_effects = [
    ChoiceLatentEffect(
        "B",
        "ATTITUDE",
        Beta("B_ATTITUDE", init=0.30),
    )
]
indicators = [
    ContinuousIndicator(
        "attitude",
        "ATTITUDE",
        intercept=0.0,
        loading=1.0,
        sigma_init=0.70,
        sigma_fixed=True,
    )
]

model = HybridChoiceModel(
    spec,
    latent_variables=latent_variables,
    choice_effects=choice_effects,
    indicators=indicators,
    n_draws=24,
    seed=22,
    antithetic=True,
    panel=False,
    device=device,
    max_iter=60,
)
result = model.fit(data)
display(HTML(result.report().to_html()))


Model family,HybridChoiceModel
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,60
Line search,strong_wolfe
